# Muon Catalysis & Tritium Fuel Cycle

The **muon recycling trifecta** targets $N_{fus,\mu} > 284$ catalytic fusions per muon (breakeven).

## Extended rate network

Five populations: free $\mu$, $d\mu$, $t\mu$, $dt\mu$ molecule, and stuck $(\alpha\mu)^+$.

Stripping rates $R = R_{col} + R_{photon} + R_{proton}$ reduce effective sticking $\omega_{eff} = \omega_0(1 - R)$.

Photon stripping (XFEL + vortex OAM) and proton collision stripping are tunable via `RateNetworkParams`.

---

## Coupled rate-network ODEs (VISION §4)

$$\frac{dN_\mu}{dt} = S_{source} + R_{strip}\, N_{\alpha\mu} - \lambda_{form} N_\mu - \frac{N_\mu}{\tau_\mu}$$

$$\frac{dN_{d\mu}}{dt} = \lambda_{form} N_\mu - \lambda_{transfer} N_{d\mu}$$

$$\frac{dN_{t\mu}}{dt} = \lambda_{transfer} N_{d\mu} - \lambda_{t\mu} N_{t\mu}$$

$$\frac{dN_{dt\mu}}{dt} = \lambda_{t\mu} N_{t\mu} + \cdots - \lambda_{fusion} N_{dt\mu}$$

$$\frac{dN_{\alpha\mu}}{dt} = \lambda_{fusion} N_{dt\mu}\, \omega_0 - R_{strip} N_{\alpha\mu} - R_{col} N_{\alpha\mu}$$

**Composite stripping** (trifecta):
$$R_{strip} = R_{col} + R_{photon} + R_{proton} + R_{cyclotron}$$

**Effective sticking** (reduced by active stripping):
$$\omega_{eff} = \omega_0 (1 - R_{strip})$$

### Breakeven proof sketch ($N_{fus,\mu} > 284$)

Each catalytic fusion releases $E_{fus} \approx 17.6$ MeV. Muon production costs $E_\mu \approx 4.1$ GeV. Energy breakeven requires:
$$N_{fus,\mu} > \frac{E_\mu}{E_{fus}} \approx \frac{4.1 \times 10^9}{17.6 \times 10^6} \approx 233$$

Including sticking losses ($\omega_0 \approx 0.008$) and transport inefficiency pushes the **operational threshold to ~284** — implemented as `BREAKEVEN_FUSIONS` in code.

| Stripping configuration (VISION) | Projected $N_{fus,\mu}$ |
|----------------------------------|--------------------------|
| Collision-only baseline | 112.6 |
| + X-ray photoelectric | 156.5 |
| + Vortex OAM unscrewing | 220.0 |
| + Proton collision | 290.0 |
| + Cyclotron resonance | 350+ |


In [ ]:
import sys
from pathlib import Path

from deepiri_fuselk.data.notebook_loaders import resolve_repo_root

repo = resolve_repo_root()

try:
    import deepiri_fuselk  # noqa: F401
except ImportError:
    sys.path.insert(0, str(repo / "src"))
    import deepiri_fuselk  # noqa: F401

import matplotlib.pyplot as plt
import numpy as np

plt.style.use("seaborn-v0_8-whitegrid")
%config InlineBackend.figure_formats = ["retina"]

from deepiri_fuselk.data.imas_loader import load_imas_hdf5
from deepiri_fuselk.data.notebook_loaders import (
    ensure_fetched_data,
    imas_to_synthetic_shot,
    list_shots,
    load_corpus_arrays,
    load_odl_meta,
    manifest_summary,
    resolve_data_root,
)

data_root = ensure_fetched_data(resolve_data_root(repo), n_shots=100, max_odl=50)
cmod_shots = list_shots(data_root, source="cmod")
syn_shots = list_shots(data_root, source="synthetic")
corpus = load_corpus_arrays(data_root)
manifest = manifest_summary(data_root)

print(f"Data lake: {data_root}")
print(f"  C-Mod (ODL) shots: {len(cmod_shots)}")
print(f"  Synthetic shots:   {len(syn_shots)}")
print(f"  ELM corpus frames: {len(corpus['labels'])} (elm_rate={corpus['elm_rate']:.2f})")
for row in manifest:
    print(f"  [{row['source']}] {row['status']} — {row['shots']} shots @ {row['fetched_at'][:19]}")


In [ ]:
from deepiri_fuselk.muon import BREAKEVEN_FUSIONS, RateNetworkParams, run_rate_network

baseline = run_rate_network(params=RateNetworkParams(R_photon=0, R_proton=0))
boosted = run_rate_network(params=RateNetworkParams(R_photon=0.5, R_proton=0.3))

print(f"Breakeven threshold: {BREAKEVEN_FUSIONS:.0f} fusions/μ")
print(f"Baseline:  {baseline.fusions_per_muon:.1f} fusions/μ  breakeven={baseline.breakeven}")
print(f"Boosted:   {boosted.fusions_per_muon:.1f} fusions/μ  breakeven={boosted.breakeven}")
print(f"Effective sticking: {boosted.effective_sticking:.4f}, strip rate: {boosted.strip_rate:.2f}")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for label, res in [("baseline", baseline), ("boosted", boosted)]:
    t_us = res.t * 1e6
    axes[0].plot(t_us, res.populations[3], label=f"{label} dtμ")
axes[0].set_xlabel("time [μs]"); axes[0].set_ylabel("dtμ population")
axes[0].legend(); axes[0].set_title("Fusion molecule buildup")

for label, res in [("baseline", baseline), ("boosted", boosted)]:
    for i, name in enumerate(res.population_names):
        axes[1].plot(res.t * 1e6, res.populations[i], alpha=0.7, label=f"{label} {name}")
axes[1].set_xlabel("time [μs]"); axes[1].set_yscale("log")
axes[1].set_title("Full rate network"); axes[1].legend(fontsize=7, ncol=2)

plt.tight_layout()
plt.show()


## Experiment: photon strip rate sweep

In [ ]:
photon_rates = np.linspace(0, 0.8, 17)
fpm = []

for r in photon_rates:
    res = run_rate_network(params=RateNetworkParams(R_photon=r, R_proton=0.2))
    fpm.append(res.fusions_per_muon)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(photon_rates, fpm, "o-", color="C5")
ax.axhline(BREAKEVEN_FUSIONS, color="red", ls="--", label="breakeven = 284")
ax.set_xlabel(r"$R_{photon}$"); ax.set_ylabel("fusions per muon")
ax.set_title("XFEL photon stripping sensitivity")
ax.legend()
plt.tight_layout()
plt.show()


## Coupling to breeding blanket (oil–water PDE)

In [ ]:
from deepiri_fuselk.barrier.breeding_blanket import evaluate_breeding_blanket
from deepiri_fuselk.physics import PDEParameters, solve_oil_water_steady

pde = solve_oil_water_steady(n_grid=64)
blanket = evaluate_breeding_blanket(pde.state, PDEParameters())

muon_ok = boosted.breakeven
fuel_ok = blanket.tritium_breeding_ratio >= 1.0

print(f"TBR = {blanket.tritium_breeding_ratio:.3f}")
print(f"Muon breakeven: {muon_ok}, Tritium self-sufficient: {fuel_ok}")
print(f"Combined fuel-cycle readiness: {muon_ok and fuel_ok}")


## Real discharge density → PDE fuel-cycle context

In [ ]:
from deepiri_fuselk.muon.stripping_orchestrator import run_stripping_trifecta

# Anchor blanket evaluation to mean ne from all fetched C-Mod shots
ne_samples = [float(np.mean(load_imas_hdf5(p).ne_profile.values)) for p in cmod_shots]
ne_mean = float(np.mean(ne_samples))

p_real = PDEParameters(n0=ne_mean * 1e-6, v_v=0.6)
pde_real = solve_oil_water_steady(n_grid=64, params=p_real)
blanket_real = evaluate_breeding_blanket(pde_real.state, p_real)
trifecta = run_stripping_trifecta()

fig, ax = plt.subplots(figsize=(8, 4))
bars = {
    "TBR (C-Mod ne)": blanket_real.tritium_breeding_ratio,
    "Pe": peclet_criterion(p_real),
    "μ fusions/μ / 284": trifecta.projected_fpm / 284.0,
    "ODL precursor rate": float(np.mean([load_odl_meta(p)["elm_event_rate"] for p in cmod_shots[:5]])),
}
ax.barh(list(bars.keys()), list(bars.values()), color=["C2", "C0", "C5", "C3"])
ax.axvline(1.0, color="gray", ls="--")
ax.set_xlim(0, 1.4)
ax.set_title("Cross-domain fuel-cycle readiness on real density scale")
plt.tight_layout()
plt.show()

print(f"Mean C-Mod ne: {ne_mean:.2e} m⁻³ → TBR={blanket_real.tritium_breeding_ratio:.3f}, Pe={peclet_criterion(p_real):.2f}")
print(f"Trifecta μ breakeven={trifecta.breakeven}, combined ready={blanket_real.tritium_breeding_ratio >= 1 and trifecta.breakeven}")


## Stripping trifecta orchestrator

In [ ]:
from deepiri_fuselk.muon.cyclotron_resonance import CyclotronConfig, cyclotron_frequency, resonance_match
from deepiri_fuselk.muon.photon_stripper import PhotonStripperConfig, stripping_rate as photon_rate
from deepiri_fuselk.muon.proton_stripper import ProtonStripperConfig, stripping_rate as proton_rate
from deepiri_fuselk.muon.stripping_orchestrator import StrippingTrifectaConfig, run_stripping_trifecta

cyc = CyclotronConfig()
print(f"Cyclotron f_c = {cyclotron_frequency(cyc):.3e} Hz, locked = {resonance_match(cyc)}")

trifecta = run_stripping_trifecta()
print(f"R_photon={trifecta.R_photon:.3f}, R_proton={trifecta.R_proton:.3f}, R_cyclotron={trifecta.R_cyclotron:.3f}")
print(f"Projected fusions/μ = {trifecta.projected_fpm:.1f}, breakeven={trifecta.breakeven}, margin={trifecta.margin_to_breakeven:.1f}")
